# Partially isophase electro-magnetic configurations — numerical companion

All calculations and figures of the paper, in one notebook (runs top to bottom in Colab / Jupyter; needs only `numpy`, `scipy`, `matplotlib`).

| Part | Paper | Content |
|---|---|---|
| 0 | – | setup, switches |
| I | Sec. II | rigid spectral flow: trajectories in the $(B_z,\lambda)$ plane, level flow (Figs. 1–2) |
| II | Sec. VI | classical orbits before/after the transformation (Fig. 6) |
| III | App. A | explicit two-channel isospectral example (Figs. A1–A3) |
| IV | App. B | cartesian slab, **resonant magnetic barrier** on an infinite strip: invariance check, fields, resonance flow, $T(E,k_y)$ maps, resonant beam, gate tolerance |

**Conventions.** Parts I–III as in the paper. Part IV: $\hbar=m=e=1$, length unit $\ell$; $V_{k_y}(x)=\tfrac12(k_y-A_y)^2+e\phi$; deformation $\tilde A_y=(1+a)A_y$, $e\tilde\phi=a\,k_{y1}A_y-(a+a^2/2)A_y^2$ with $a\equiv$ `amp`.

**Run time.** Everything is a few minutes on a laptop; the heaviest cells are the $T(E,k_y)$ maps (set `FAST=True` for a coarse grid) and the classical orbits.

**Note.** Figures 1–2 (Part I) were re-written for this notebook from the formulas in the paper (the original scripts were not kept), so their look differs slightly from the paper figures; the content is the same.

In [ ]:
import numpy as np, time
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LogNorm, PowerNorm
from scipy.integrate import solve_ivp, cumulative_trapezoid
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

SAVE = False      # True: write PNG/PDF files next to the notebook
FAST = False      # True: coarse grids for the (E,k_y) maps (quick look)
plt.rcParams.update({"font.size": 10.5, "mathtext.fontset": "cm", "axes.linewidth": 1.0})
C_BEFORE, C_AFTER = "0.55", "#c53030"

def save(fig, name, dpi=170):
    if SAVE:
        fig.savefig(name + ".png", dpi=dpi, bbox_inches="tight"); fig.savefig(name + ".pdf", bbox_inches="tight")

# Part I — rigid spectral flow (Sec. II)

For $f=b\rho^2/2$ and $E^{(0)}_\rho=\lambda\rho$ the transformation $b\to b+\delta b$, $\lambda\to\lambda+\frac{e\,\delta b}{4m}(2b+\delta b)$ keeps $\Omega^2=\omega_c^2/4-e\lambda/m$ fixed, i.e. it moves along parabolas $\lambda=B_z^2/4-\Omega^2$ (units $e=m=\hbar=1$). The unprotected channels $l'$ shift linearly, $\delta E_{n,l'}=e\hbar(l-l')\delta b/2m$, on top of an arbitrary base spectrum.

In [ ]:
# ---- Fig. 1: trajectories in the (B_z, lambda) plane ----
Bz = np.linspace(-4.2, 4.2, 400)
fig, ax = plt.subplots(figsize=(6.6, 5.4))
Om = np.array([0.5, 0.8, 1.2, 1.5, 2.0, 2.5, 3.0])
cm = plt.get_cmap("viridis"); nrm = plt.Normalize(0.4, 3.0)
ax.fill_between(Bz, Bz**2/4, 5.4, color="0.88", zorder=0)               # Omega^2 < 0
ax.plot(Bz, Bz**2/4, "k--", lw=1.2, alpha=0.8)
for w in Om: ax.plot(Bz, Bz**2/4 - w**2, color=cm(nrm(w)), lw=2.4)
b0, w0, db = 1.0, 1.5, 2.0
l0 = b0**2/4 - w0**2; l1 = (b0+db)**2/4 - w0**2                          # one step b -> b+db
ax.annotate("", xy=(b0+db, l1), xytext=(b0, l0), arrowprops=dict(arrowstyle="-|>", lw=2, color="k"))
ax.plot([b0, b0+db], [l0, l1], "ko", ms=6)
ax.text(b0-0.05, l0-0.7, r"$(b,\lambda)$", fontsize=11)
ax.text(b0+db-0.1, l1+0.5, r"$(b+\delta b,\lambda+\delta\lambda)$", fontsize=10)
ax.set_xlim(-4.2, 4.2); ax.set_ylim(-9.5, 5.2)
ax.set_xlabel(r"$B_z$"); ax.set_ylabel(r"$\lambda\equiv E_\rho^{(0)}/\rho$")
sm = plt.cm.ScalarMappable(norm=nrm, cmap=cm); fig.colorbar(sm, ax=ax).set_label(r"spectral quantum $\hbar\Omega$ (invariant along each curve)")
fig.tight_layout(); save(fig, "fig1_isophase_trajectories"); plt.show()

In [ ]:
# ---- Fig. 2: universal linear flow of the levels on top of a non-equidistant base spectrum ----
xi = np.linspace(0, 5, 50)                     # xi = -delta b
E0 = np.array([1.0, 2.3, 3.3, 5.0])            # arbitrary (non-equidistant) bare levels of the protected channel l=0
dE = {0: 0.0, 1: 0.5, 2: 1.0}                  # delta E_{n,l} = (e hbar l / 2m) xi,  here l*0.5
off = {0: 0.0, 1: 0.6, 2: 1.2}                 # bare offset of the unprotected channels (arbitrary)
fig, ax = plt.subplots(figsize=(7.2, 5.4))
col = {0: "0.15", 1: "#2b6cb0", 2: "#c53030"}
for l in (0, 1, 2):
    for n, e0 in enumerate(E0):
        ax.plot(xi, e0 + off[l] + dE[l]*xi, color=col[l], lw=2.6 if l == 0 else 2.2)
for n, e0 in enumerate(E0): ax.text(5.08, e0, rf"$n={n}$", ha="left", va="center", fontsize=9)
ax.text(4.1, 11.1, r"$l=2$", color=col[2]); ax.text(4.1, 8.4, r"$l=1$", color=col[1]); ax.text(0.1, 11.3, r"$l=0$ (protected: flat)", color=col[0])
ax.set_xlim(-0.2, 5.6); ax.set_xlabel(r"$\xi=-\delta b$ (flow parameter)"); ax.set_ylabel(r"$E_{n,l}$")
fig.tight_layout(); save(fig, "fig2_rigid_flow_levels"); plt.show()

# Part II — classical orbits (Sec. VI)

Radial Hamiltonian $H_L=p_\rho^2/2m+(L-ef)^2/2m\rho^2+e\phi$. After $f\to f+\delta f$, $e\phi\to e\phi-[2\delta f(ef-L_0)+e\delta f^2]/2m\rho^2$ the radial motion at $L=L_0$ is unchanged, while the azimuth acquires $-\int\Omega_\delta\,dt$, $\Omega_\delta=e\,\delta f/m\rho^2$. The orbits are obtained by integrating Newton's equations with the Lorentz force in the plane.

In [ ]:
e = m = 1.0
def d1(fun, r, h=1e-6): return (fun(r + h) - fun(r - h))/(2*h)

def make_case(f0, phi0, df, L0):
    ft = lambda r: f0(r) + df(r)
    def phit(r):
        d = df(r); return phi0(r) - (2*d*(e*f0(r) - L0) + e*d**2)/(2*m*r**2)
    return ft, phit

def rhs_factory(f, phi):
    def rhs(t, y):
        x, yy, vx, vy = y; r = np.hypot(x, yy)
        Bz, Er = d1(f, r)/r, -d1(phi, r)
        return [vx, vy, e*(Er*x/r + vy*Bz)/m, e*(Er*yy/r - vx*Bz)/m]
    return rhs

def run_orbit(f, phi, L0, r0, pr, T, n=12000):
    phid = (L0 - e*f(r0))/(m*r0**2)
    t = np.linspace(0, T, n)
    s = solve_ivp(rhs_factory(f, phi), [0, T], [r0, 0, pr, r0*phid], t_eval=t, rtol=1e-11, atol=1e-12)
    x, y = s.y[0], s.y[1]
    return t, x, y, np.hypot(x, y), np.unwrap(np.arctan2(y, x))

def orbit_case(f0, phi0, df, L0, r0, pr, T):
    ft, phit = make_case(f0, phi0, df, L0)
    a = run_orbit(f0, phi0, L0, r0, pr, T); c = run_orbit(ft, phit, L0, r0, pr, T)
    Om = e*df(a[3])/(m*a[3]**2)
    pred = -cumulative_trapezoid(Om, a[0], initial=0)
    return a, c, pred

# case 1: Fock-Darwin -> Fock-Darwin (rigid, delta f = rho^2 db/2); w+/w- = 3 -> 2
b, lam, db = 1.0, -0.75, -1.0/3
c1 = orbit_case(lambda r: b*r**2/2, lambda r: -lam*r**2/2, lambda r: r**2*db/2, L0=-1.0, r0=1.3, pr=0.4, T=4*np.pi)
# case 2: smooth B(rho), E0 = 0, generic delta f
b0_, b1_, w_ = 0.6, 1.4, 1.3
f0_2 = lambda r: b0_*r**2/2 + b1_*w_**2/2*(1 - np.exp(-r**2/w_**2))
df_2 = lambda r: 0.5*r**2*np.exp(-r**2/2)
c2 = orbit_case(f0_2, lambda r: 0*r, df_2, L0=-0.8, r0=1.0, pr=0.5, T=40.0)
for name, (a, c, pred) in (("Fock-Darwin rigid", c1), ("smooth B(rho)", c2)):
    print(f"{name}: max|rho_after-rho_before| = {np.max(np.abs(a[3]-c[3])):.2e};  max|dphi - prediction| = {np.max(np.abs((c[4]-a[4])-pred)):.2e}")

In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
fig = plt.figure(figsize=(13.6, 8.6))
gs = GridSpec(2, 3, figure=fig, width_ratios=[1, 1, 1.25], hspace=0.28, wspace=0.28)
th = np.linspace(0, 2*np.pi, 400)
rows = [("Fock–Darwin, rigid deformation $\\delta f=\\rho^2\\delta b/2$", c1), ("smooth $B_z(\\rho)$, $E^{(0)}=0$, generic $\\delta f$", c2)]
for i, (title, (a, c, pred)) in enumerate(rows):
    rmin, rmax = a[3].min(), a[3].max()
    for j, (d, col, ttl) in enumerate([(a, C_BEFORE, "before"), (c, C_AFTER, "after")]):
        ax = fig.add_subplot(gs[i, j])
        for rr in (rmin, rmax): ax.plot(rr*np.cos(th), rr*np.sin(th), color="0.75", lw=0.9, ls=":")
        ax.plot(d[1], d[2], color=col, lw=1.0); ax.plot([d[1][0]], [d[2][0]], "o", color="k", ms=5, zorder=5); ax.plot([0], [0], "+", color="k", ms=8)
        ax.set_aspect("equal"); lim = rmax*1.12; ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_xlabel("$x$"); ax.set_ylabel("$y$"); ax.set_title(ttl, fontsize=10.5)
        if j == 0: ax.text(0.0, 1.13, title, transform=ax.transAxes, fontsize=11.5, fontweight="bold")
    sub = GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[i, 2], hspace=0.15)
    ax1 = fig.add_subplot(sub[0]); ax2 = fig.add_subplot(sub[1], sharex=ax1)
    ax1.plot(a[0], a[3], color=C_BEFORE, lw=4.5, alpha=0.55, label="before"); ax1.plot(c[0], c[3], color=C_AFTER, lw=1.4, ls="--", label="after")
    ax1.set_ylabel(r"$\rho(t)$"); ax1.legend(frameon=False, ncol=2, loc="upper center", fontsize=9)
    ax1.set_ylim(rmin-0.04*(rmax-rmin), rmax+0.32*(rmax-rmin)); plt.setp(ax1.get_xticklabels(), visible=False)
    ax2.plot(a[0], c[4]-a[4], color=C_AFTER, lw=2.0, label=r"$\varphi_{\rm after}-\varphi_{\rm before}$")
    ax2.plot(a[0], pred, color="k", lw=1.0, ls=":", label=r"$-\int\Omega_\delta(\rho(t'))\,dt'$")
    ax2.set_ylabel(r"$\Delta\varphi(t)$"); ax2.set_xlabel("$t$"); ax2.legend(frameon=False, fontsize=8.5, loc="best"); ax2.set_xlim(0, a[0][-1])
save(fig, "fig_classical_orbits2", 200); plt.show()

# Part III — Appendix A: explicit two-channel isospectral example

Merged from the stand-alone notebook `two_channel_isophase_example.ipynb` (unchanged code).

Companion notebook to the Appendix of *Partially isophase electro-magnetic configurations for vortex electrons*.

**Seed background.** Uniform magnetic field $B_z=\omega$ plus an Aharonov–Bohm flux $\mu=1/2$ (in flux quanta), no electric field.
Units: $\hbar^2/2m=1$, $F(\rho)=e f(\rho)/\hbar=\mu+\omega\rho^2/2$ (flux in units of the flux quantum), radial equation $-u''+V_l u=Eu$ with

$$V_l(\rho)=\frac{(F-l)^2-1/4}{\rho^2}+e\phi ,\qquad V_{0}=\frac{\omega}{2}+\frac{\omega^2\rho^2}{4},\quad V_{1}=-\frac{\omega}{2}+\frac{\omega^2\rho^2}{4}.$$

Because $|l-\mu|=1/2$ for $l=0,1$, the centrifugal term vanishes and both channels are oscillators on the half-line ($u(0)=0$).

**Construction.**
* protected channel (exact partial isophase): $l=1$;  isospectral (Darboux) channel: $l_s=0$;  unprotected test channel: $l'=2$;
* two Crum seeds of the $l_s=0$ channel, both *below* the ground state $2\omega$:
  $u_1=\rho\,e^{-\omega\rho^2/4}\,{}_1F_1(a_1;3/2;\omega\rho^2/2)$ (regular at 0), $u_2=e^{-\omega\rho^2/4}\,U(a_2;1/2;\omega\rho^2/2)$ (decaying), with $a_1=1-E_1/2\omega$, $a_2=1/2-E_2/2\omega$;
* $W=\mathrm{Wr}[u_1,u_2]$, $\delta F=\rho^2(\ln W)''/(l_s-l)$ (Eq. 18), $e\tilde\phi=-[2\delta F(F-l)+\delta F^2]/\rho^2$ (Eq. 6), $\Delta V_{l'}=2(l-l')\,\delta F/\rho^2$ (Eq. 7).

> **Note on the factorisation constants.** $E_1,E_2$ here are the *true energies* of $V_{l_s}$ (the $\alpha_j$ of Eq. 20 of the paper). In the original Mathematica notebook the constants $\alpha_1=0,\ \alpha_2=-2$ are measured from the bare oscillator, i.e. $E=\alpha+\omega/2$, so the same example is $E_1=1/2,\ E_2=-3/2$ (default below).

Everything except the spectra is evaluated analytically: $W'=(E_1-E_2)u_1u_2$ is an exact identity, so $(\ln W)''$ and $(\ln W)'''$ need no numerical differentiation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import hyp1f1, hyperu
from scipy.linalg import eigh_tridiagonal

In [ ]:
# ----------------------------------------------------------------------------
# parameters (edit / experiment)
# ----------------------------------------------------------------------------
P = dict(
    omega=1.0,          # uniform-field strength  (B_z^(0) = omega)
    mu=0.5,             # AB flux, in flux quanta  (must be 1/2 for this closed-form seed)
    ls=0, l=1,          # isospectral channel, protected channel  (nu = |l - mu| = 1/2 for both)
    E1=0.5, E2=-1.5,    # factorisation energies, both below the ground state 2*omega
    lprime=2,           # an additional, unprotected channel
    R=16.0, N=8000,     # box size and number of grid points for the finite-difference solver
    nlev=10,            # number of computed levels
)

In [ ]:
# ----------------------------------------------------------------------------
# seeds, Wronskian and derivatives of ln W (all analytic)
# ----------------------------------------------------------------------------
def seeds(r, p=P):
    om, E1, E2 = p["omega"], p["E1"], p["E2"]
    z = om * r**2 / 2
    a1 = 1.0 - E1 / (2 * om)          # = 3/4 - (E1 - om/2)/(2 om)
    a2 = 0.5 - E2 / (2 * om)          # = 1/4 - (E2 - om/2)/(2 om)
    ex = np.exp(-om * r**2 / 4)
    M, M1 = hyp1f1(a1, 1.5, z), hyp1f1(a1 + 1, 2.5, z)
    U, U1 = hyperu(a2, 0.5, z), hyperu(a2 + 1, 1.5, z)
    u1 = r * ex * M
    du1 = ex * M * (1 - om * r**2 / 2) + r * ex * (a1 / 1.5) * M1 * om * r
    u2 = ex * U
    du2 = ex * (-om * r / 2 * U - a2 * om * r * U1)
    return u1, du1, u2, du2


def V_seed(ell, r, p=P):
    """V_l of the seed background (F = mu + om r^2/2, phi = 0)."""
    F = p["mu"] + p["omega"] * r**2 / 2
    return ((F - ell)**2 - 0.25) / r**2


def lnW_derivs(r, p=P):
    """W = Wr[u1,u2] and L2 = (ln W)'', L3 = (ln W)'''  (no numerical differentiation)."""
    E1, E2 = p["E1"], p["E2"]
    u1, du1, u2, du2 = seeds(r, p)
    W = u1 * du2 - du1 * u2
    V0 = V_seed(p["ls"], r, p)
    d1, d2 = (V0 - E1) * u1, (V0 - E2) * u2         # u'' from the Schroedinger equation
    c = E1 - E2
    Wp, Wpp = c * u1 * u2, c * (du1 * u2 + u1 * du2)  # W' = (E1-E2) u1 u2 (exact identity)
    Wppp = c * (d1 * u2 + 2 * du1 * du2 + u1 * d2)
    q = Wp / W
    L2 = Wpp / W - q**2
    L3 = Wppp / W - q * Wpp / W - 2 * q * L2
    return W, L2, L3


# ----------------------------------------------------------------------------
# deformed background:  delta F, phi~, fields, potentials
# ----------------------------------------------------------------------------
def deformation(r, p=P):
    om, mu, ls, l = p["omega"], p["mu"], p["ls"], p["l"]
    W, L2, L3 = lnW_derivs(r, p)
    dF = r**2 * L2 / (ls - l)                       # Eq. (18) in these units
    dFp = (2 * r * L2 + r**2 * L3) / (ls - l)       # d(dF)/dr
    F = mu + om * r**2 / 2
    g = 2 * dF * (F - l) + dF**2                    # Eq. (6):  e*phi~ = -g / rho^2
    gp = 2 * dFp * (F - l) + 2 * dF * om * r + 2 * dF * dFp
    return dict(W=W, L2=L2, L3=L3, dF=dF, dFp=dFp, F=F, g=g, gp=gp,
                ephi=-g / r**2,
                Bz0=om + 0 * r, Bzt=om + dFp / r,
                Er0=0 * r, Ert=gp / r**2 - 2 * g / r**3,
                dV_over_dl=2 * dF / r**2)


def V_def(ell, r, p=P):
    """Deformed radial potential  V~_l = ((F+dF-l)^2 - 1/4)/r^2 + e*phi~."""
    d = deformation(r, p)
    return ((d["F"] + d["dF"] - ell)**2 - 0.25) / r**2 + d["ephi"]


# ----------------------------------------------------------------------------
# numerical spectra and eigenfunctions   -u'' + V u = E u ,  u(0)=u(R)=0
# ----------------------------------------------------------------------------
def solve(Vfun, p=P, k=None):
    R, N, k = p["R"], p["N"], (k or p["nlev"])
    h = R / N
    x = h * np.arange(1, N)
    d = 2 / h**2 + Vfun(x)
    e = -np.ones(N - 2) / h**2
    E, v = eigh_tridiagonal(d, e, select="i", select_range=(0, k - 1))
    u = v / np.sqrt(h)                                # int u^2 dr = 1
    for j in range(k):                                # sign: first lobe positive
        i0 = np.argmax(np.abs(u[:, j]) > 0.05 * np.abs(u[:, j]).max())
        u[:, j] *= np.sign(u[i0, j])
    return x, E, u


# ----------------------------------------------------------------------------
# independent check: Crum formula  u~_n = Wr[u1,u2,u_n] / Wr[u1,u2]
# ----------------------------------------------------------------------------
def oscillator_state(n, r, p=P):
    """exact eigenfunctions of V_{ls} (E_n = 2*omega*(n+1)), u ~ r exp(-om r^2/4) 1F1(-n,3/2,om r^2/2)."""
    om = p["omega"]
    z = om * r**2 / 2
    ex = np.exp(-om * r**2 / 4)
    M = hyp1f1(-n, 1.5, z)
    dM = (-n / 1.5) * hyp1f1(-n + 1, 2.5, z) if n > 0 else 0 * r
    u = r * ex * M
    du = ex * M * (1 - om * r**2 / 2) + r * ex * dM * om * r
    return u, du, 2 * om * (n + 1)


def crum_state(n, r, p=P):
    u1, du1, u2, du2 = seeds(r, p)
    un, dun, En = oscillator_state(n, r, p)
    V0 = V_seed(p["ls"], r, p)
    ddu1, ddu2, ddun = (V0 - p["E1"]) * u1, (V0 - p["E2"]) * u2, (V0 - En) * un
    W3 = (u1 * (du2 * ddun - dun * ddu2)
          - u2 * (du1 * ddun - dun * ddu1)
          + un * (du1 * ddu2 - du2 * ddu1))
    W2 = u1 * du2 - du1 * u2
    ut = W3 / W2
    h = r[1] - r[0]
    ut = ut / np.sqrt(np.sum(ut**2) * h)
    return ut

### Sanity checks
Nodelessness of $W$, the shift formula, and the asymptotics of the deformation.

In [ ]:
r = np.linspace(1e-3, 12, 6001)
d = deformation(r)
print("W nodeless on (0,12]:", bool(np.all(d["W"] < 0)), "   max W =", d["W"].max())
# Eq. (7): V~_l' - V_l' = (l - l') * 2 dF / rho^2 for any l'
x = np.linspace(0.2, 6, 50); dx = deformation(x)
for lp in (0, 1, 2, 3, -1):
    err = np.max(np.abs(V_def(lp, x) - V_seed(lp, x) - (P["l"] - lp) * dx["dV_over_dl"]))
    print(f"l'={lp:>2d}:  max |Delta V - (l-l') 2dF/rho^2| = {err:.1e}")
far = deformation(np.array([20.0, 30.0]))
print("dF(20), dF(30)   ->", far["dF"], "  (tends to -2: deformation carries a net flux of -2 flux quanta)")
print("e*phi~(20,30)    ->", far["ephi"], "  (tends to +2: constant shift of the scalar potential at infinity)")
print("B~_z(20,30) - omega ->", far["Bzt"] - P["omega"])
near = deformation(np.array([1e-3]))
print("rho->0+:  B~_z =", near["Bzt"][0], "  E~_rho =", near["Ert"][0], "  (finite, with a cusp ~ |rho|; AB flux mu unchanged)")

In [ ]:
plt.rcParams.update({"font.size": 11, "mathtext.fontset": "cm", "axes.linewidth": 1.0})

C_LS, C_L, C_L2 = "#2b6cb0", "0.12", "#c53030"      # isospectral (blue), protected (dark), unprotected (red)
C_BEFORE = "0.55"


def compute(p=P):
    out = {}
    chans = {"ls": p["ls"], "l": p["l"], "lp": p["lprime"]}
    for key, ell in chans.items():
        x, E0, u0 = solve(lambda t, ell=ell: V_seed(ell, t, p), p)
        _, E1, u1 = solve(lambda t, ell=ell: V_def(ell, t, p), p)
        out[key] = dict(ell=ell, x=x, E0=E0, u0=u0, E1=E1, u1=u1)
    return out


def validation(res, p=P):
    print("=== validation ===")
    for key in ("ls", "l", "lp"):
        d = res[key]
        print(f"channel {key:>2s} (l={d['ell']}):  max|E_after - E_before| over {len(d['E0'])} levels = "
              f"{np.max(np.abs(d['E1'] - d['E0'])):.2e}")
    print("first levels, l'=%d  before:" % p["lprime"], np.round(res["lp"]["E0"][:6], 4))
    print("                      after :", np.round(res["lp"]["E1"][:6], 4))
    print("                      shift :", np.round(res["lp"]["E1"][:6] - res["lp"]["E0"][:6], 4))
    # protected channel: eigenfunctions identical?
    du = np.max(np.abs(np.abs(res["l"]["u1"][:, :5]) - np.abs(res["l"]["u0"][:, :5])))
    print(f"protected l={p['l']}: max eigenfunction difference (first 5) = {du:.2e}")
    # Crum formula vs. numerical eigenfunctions of the deformed l_s channel
    x = res["ls"]["x"]
    for n in range(3):
        uc = crum_state(n, x, p)
        un = res["ls"]["u1"][:, n]
        uc *= np.sign(uc[np.argmax(np.abs(uc) > 0.05 * np.abs(uc).max())])
        print(f"Crum u~_{n} vs numerical eigenfunction of V~_{p['ls']}: max diff = {np.max(np.abs(uc - un)):.2e}")


def fig_fields(p=P, name="ex_fields"):
    r = np.linspace(0.004, 6.0, 3000)
    d = deformation(r, p)
    fig, ax = plt.subplots(1, 3, figsize=(12.2, 3.7))

    ax[0].plot(r, d["Bz0"], color=C_BEFORE, ls="--", lw=2, label="before")
    ax[0].plot(r, d["Bzt"], color=C_LS, lw=2.4, label="after")
    ax[0].set_ylabel(r"$eB_z/\hbar$")
    ax[0].set_title(r"(a) magnetic field, regular part", fontsize=11)
    ax[0].legend(frameon=False, loc="upper right")

    ax[1].plot(r, d["Er0"], color=C_BEFORE, ls="--", lw=2)
    ax[1].plot(r, d["Ert"], color=C_L2, lw=2.4)
    ax[1].set_ylabel(r"$eE_\rho$  [$\hbar^2/2m\ell$]")
    ax[1].set_title(r"(b) radial electric field", fontsize=11)

    ax[2].axhline(0, color="0.7", lw=0.8)
    ax[2].plot(r, d["dV_over_dl"], color="#2f855a", lw=2.4)
    ax[2].set_ylabel(r"$\Delta V_{l'}/(l-l')$  [$\hbar^2/2m\ell^2$]")
    ax[2].set_title(r"(c) potential deformation, $=2\,\delta F/\rho^2$", fontsize=11)

    for a in ax:
        a.set_xlabel(r"$\rho$  [$\ell$]")
        a.set_xlim(0, 6)
        a.axhline(0, color="0.85", lw=0.6, zorder=0)
    fig.tight_layout()
    if SAVE:
        fig.savefig(name + ".png", dpi=200)
        fig.savefig(name + ".pdf")
    plt.show()


def fig_channels(res, p=P, name="ex_channels", nwf=4, nlines=6, scale=0.95):
    panels = [("ls", C_LS, r"$l_s=%d$: isospectral (Darboux)" % p["ls"]),
              ("l", C_L, r"$l=%d$: protected (exact partial isophase)" % p["l"]),
              ("lp", C_L2, r"$l=%d$: unprotected" % p["lprime"])]
    xr = np.linspace(0.03, 6.5, 2500)
    fig, axs = plt.subplots(1, 3, figsize=(13.2, 5.0), sharey=True)
    for a, (key, col, title) in zip(axs, panels):
        d = res[key]
        ell = d["ell"]
        V0 = V_seed(ell, xr, p)
        V1 = V_def(ell, xr, p)
        a.plot(xr, V0, color=C_BEFORE, ls="--", lw=1.8, zorder=2)
        a.plot(xr, V1, color=col, lw=2.2, zorder=3)
        for n in range(nlines):
            a.axhline(d["E0"][n], color=C_BEFORE, lw=0.7, ls=(0, (3, 3)), zorder=1)
            a.axhline(d["E1"][n], color=col, lw=0.7, alpha=0.6, zorder=1)
        for n in range(nwf):
            a.plot(d["x"], d["E0"][n] + scale * d["u0"][:, n], color=C_BEFORE, ls="--", lw=1.6, zorder=4)
            a.plot(d["x"], d["E1"][n] + scale * d["u1"][:, n], color=col, lw=2.0, zorder=5)
            a.fill_between(d["x"], d["E1"][n], d["E1"][n] + scale * d["u1"][:, n], color=col, alpha=0.10, zorder=1)
        a.set_title(title, fontsize=11.5)
        a.set_xlabel(r"$\rho$  [$\ell$]")
        a.set_xlim(0, 6.5)
        a.set_ylim(-2.2, 12.0)
    axs[0].set_ylabel(r"$V_l(\rho)$, $E_n$, and $E_n+u_n(\rho)$")
    # legend
    axs[0].plot([], [], color=C_BEFORE, ls="--", lw=1.8, label="before")
    axs[0].plot([], [], color=C_LS, lw=2.2, label="after")
    axs[0].legend(frameon=False, loc="upper left", fontsize=10)
    fig.tight_layout()
    if SAVE:
        fig.savefig(name + ".png", dpi=200)
        fig.savefig(name + ".pdf")
    plt.show()


def fig_spectrum(res, p=P, name="ex_spectrum", nshow=8):
    fig, ax = plt.subplots(1, 2, figsize=(11.2, 4.4), gridspec_kw=dict(width_ratios=[1.25, 1]))
    keys = [("ls", C_LS), ("l", C_L), ("lp", C_L2)]
    for j, (key, col) in enumerate(keys):
        d = res[key]
        for n in range(nshow):
            ax[0].plot([j - 0.32, j + 0.32], [d["E0"][n]] * 2, color=C_BEFORE, ls="--", lw=1.4, zorder=1)
            ax[0].plot([j - 0.32, j + 0.32], [d["E1"][n]] * 2, color=col, lw=2.2, zorder=2)
    ax[0].set_xticks(range(3))
    ax[0].set_xticklabels([r"$l_s=%d$" % p["ls"] + "\nisospectral", r"$l=%d$" % p["l"] + "\nprotected",
                           r"$l=%d$" % p["lprime"] + "\nunprotected"])
    ax[0].set_ylabel(r"$E_n$  [$\hbar^2/2m\ell^2$]")
    ax[0].set_title("(a) spectra before (dashed) and after (solid)", fontsize=11)
    ax[0].set_xlim(-0.55, 2.55)

    n = np.arange(nshow)
    for (key, col, mk) in [("ls", C_LS, "o"), ("l", C_L, "s"), ("lp", C_L2, "D")]:
        d = res[key]
        lab = (r"$l_s=%d$" if key == "ls" else r"$l=%d$") % d["ell"]
        ax[1].plot(n, d["E1"][:nshow] - d["E0"][:nshow], mk + "-", color=col, ms=6, lw=1.8, label=lab)
    ax[1].axhline(0, color="0.7", lw=0.8)
    ax[1].set_xlabel(r"level number $n$")
    ax[1].set_ylabel(r"$\delta E_n = E_n^{\rm after}-E_n^{\rm before}$")
    ax[1].set_title("(b) spectral shifts", fontsize=11)
    ax[1].legend(frameon=False)
    fig.tight_layout()
    if SAVE:
        fig.savefig(name + ".png", dpi=200)
        fig.savefig(name + ".pdf")
    plt.show()

### Spectra and eigenfunctions (finite differences, Dirichlet at $\rho=0$ and $\rho=R$)

In [ ]:
res = compute(P)
validation(res, P)

### Figures
Set `SAVE = True` to write PNG/PDF files next to the notebook (in Colab: left sidebar → Files → download).

In [ ]:
fig_fields(P)
fig_channels(res, P)
fig_spectrum(res, P)

### Things to try
* `P["E1"], P["E2"]`: move the factorisation energies (both must stay below `2*omega`; `E1 != E2`). Larger `E1 - E2` gives a stronger deformation.
* `P["omega"]`: rescales the picture (the example is scale invariant up to $\rho\to\rho/\sqrt\omega$).
* `P["lprime"]`: other unprotected channels, e.g. `-1`, `3` (for $l'\ne0,1$ the potential keeps a centrifugal term $(\nu^2-1/4)/\rho^2$ with $\nu=|l'-\mu|$).
* `P["R"], P["N"]`: box size and grid; the isospectrality check `max|E_after - E_before|` for $l_s$ measures the discretisation error.
* The Crum formula $\tilde u_n=\mathrm{Wr}[u_1,u_2,u_n]/\mathrm{Wr}[u_1,u_2]$ is implemented in `crum_state(n, r)` and compared with the numerical eigenfunctions in `validation`.

*Numerical remark:* `scipy.special.hyperu` has a tiny (~1e-8 relative) discontinuity at some arguments (e.g. $\rho\approx2.406$ for the default parameters); it is invisible in all plots but shows up if one differentiates the fields numerically. The analytic derivatives used here avoid this.

# Part IV — Appendix B: resonant magnetic barrier on an infinite strip

Units $\hbar=m=e=1$. Gauge $A=A_y(x)\mathbf e_y$; each incidence channel sees $V_{k_y}(x)=\tfrac12(k_y-A_y)^2+e\phi$.

In [ ]:
# ---------------- barrier profile (closed form) ----------------
b, d, H = 0.18, 1.0, 1.5          # stripe strength, depth below the 2DEG, thickness
Lx, gap = 3.0, 6.0                # stripe length, centre-to-centre distance of the two stripes

def B1(x): return b*(np.log(x**2 + d**2) - np.log(x**2 + (d+H)**2))
def A1(x): return b*(x*(np.log(x**2+d**2) - np.log(x**2+(d+H)**2)) + 2*d*np.arctan(x/d) - 2*(d+H)*np.arctan(x/(d+H)))

xa1, xa2 = -gap/2 - Lx/2, -gap/2 + Lx/2        # stripe 1
xb1, xb2 = +gap/2 - Lx/2, +gap/2 + Lx/2        # stripe 2
Bz0 = lambda x: (B1(x-xa1) - B1(x-xa2)) + (B1(x-xb1) - B1(x-xb2))      # B_z(x), odd
Ay0 = lambda x: (A1(x-xa1) - A1(x-xa2)) + (A1(x-xb1) - A1(x-xb2))      # A_y(x), even, -> 0 at |x|->inf

# check A' = B and the leads
xx = np.linspace(-15, 15, 3001); h_ = xx[1]-xx[0]
print("max|A_y' - B_z| =", np.max(np.abs(np.gradient(Ay0(xx), h_) - Bz0(xx))[2:-2]), "  A_y(+-40) =", Ay0(np.array([-40., 40.])))
hbar2_me = 7.62            # eV * Angstrom^2  (hbar^2 / m_e)
ell_A = 500.0              # l = 50 nm
print("GaAs scale, m*=0.067, l=50 nm: energy unit %.3f meV, field unit %.3f T" % (1e3*hbar2_me/0.067/ell_A**2, 6.582e-16/(50e-9)**2))

### Scattering solver

Numerov integration of $-\tfrac12\chi''+V_{k_y}\chi=E\chi$ from $x=+L$ (purely outgoing wave) back to $x=-L$; vectorised over arrays of $(E,k_y,a)$. Returns the complex transmission amplitude $t$ and $T=(k_R/k_L)|t|^2$. The potential is passed as a function `V(j)` of the grid index, $x_j=L-hj$.

In [ ]:
L_, h_ = 30.0, 0.02
M_ = int(round(2*L_/h_)); xs = L_ - h_*np.arange(M_ + 1)       # decreasing grid
A0g = Ay0(xs)                                                  # A_y on the grid

def numerov(E, ky, V, want_psi=False):
    # E, ky broadcastable arrays;  V(j) -> potential at node j (broadcast with E)
    E, ky = np.broadcast_arrays(np.asarray(E, float), np.asarray(ky, float)); shp = E.shape
    E, ky = E.ravel(), ky.ravel()
    Vj = lambda j: np.broadcast_to(V(j, ky), E.shape)
    kL2, kR2 = 2*(E - Vj(M_)), 2*(E - Vj(0)); ok = (kL2 > 1e-6) & (kR2 > 1e-6)
    kL = np.sqrt(np.where(ok, kL2, 1.0)); kR = np.sqrt(np.where(ok, kR2, 1.0))
    f = lambda j: 2*(Vj(j) - E)
    p_prev = np.ones_like(E, dtype=complex); p_cur = np.exp(-1j*kR*h_)
    f_prev, f_cur = f(0), f(1); c = h_*h_/12.0
    if want_psi:
        store = np.empty((M_+1,) + E.shape, dtype=complex); store[0], store[1] = p_prev, p_cur
    for j in range(1, M_):
        f_next = f(j+1)
        p_next = (2*p_cur*(1 + 5*c*f_cur) - p_prev*(1 - c*f_prev))/(1 - c*f_next)
        p_prev, p_cur, f_prev, f_cur = p_cur, p_next, f_cur, f_next
        if want_psi: store[j+1] = p_next
    eL = np.exp(-1j*kL*L_); eLh = np.exp(1j*kL*(-L_ + h_))
    det = eL*np.conj(eLh) - np.conj(eL)*eLh
    a = (p_cur*np.conj(eLh) - np.conj(eL)*p_prev)/det
    that = np.exp(-1j*kR*L_)/a
    T = np.where(ok, (kR/kL)*np.abs(that)**2, np.nan)
    out = dict(T=T.reshape(shp), t=np.where(ok, that, np.nan).reshape(shp))
    if want_psi: out["xs"] = xs; out["psi"] = store.reshape((M_+1,) + shp)/a.reshape(shp)
    return out

# --- the isophase family on the infinite strip ---
#   A~ = (1+amp) A0,  e phi~ = amp*ky1*A0 - (amp + amp^2/2) A0^2   =>   V~_ky = (ky-A0)^2/2 + amp (ky1-ky) A0
def family(E, ky, amp, ky1, want_psi=False):
    amp = np.asarray(amp, float)
    E, ky, amp = np.broadcast_arrays(np.asarray(E, float), np.asarray(ky, float), amp)
    Vf = lambda j, kyr: 0.5*(kyr - A0g[j])**2 + amp.ravel()*(ky1 - kyr)*A0g[j]
    return numerov(E, ky, Vf, want_psi)

# sanity: free particle limit (b=0 not available) -> use total probability T+R=1 only checked in the original; here check parity/T<=1
r = family(np.linspace(0.1, 1, 5), 0.1, 0.0, 0.15); print("T(E=0.1..1, ky=0.1) =", np.round(r["T"], 4))

### 1. Exact invariance of the protected channel

For $k_y=k_{y1}$ the 1D potential and the complex amplitude $t(E)$ must coincide before/after for every $a$ (not only $T$). Here the deformed potential is built explicitly from $\tilde A_y$ and $e\tilde\phi$, without using the closed form.

In [ ]:
ky1 = 0.15
A_ = lambda amp: (1+amp)*A0g
Phi = lambda amp: amp*ky1*A0g - (amp+amp**2/2)*A0g**2
for amp in (0.5, -0.5, 1.0):
    Vb = 0.5*(ky1 - A0g)**2; Va = 0.5*(ky1 - A_(amp))**2 + Phi(amp)
    Eg_ = np.linspace(0.05, 1.0, 4000)
    # explicit (not closed-form) deformed potential: (ky - A~)^2/2 + e phi~
    r0 = numerov(Eg_, ky1, lambda j, k: 0.5*(k - A0g[j])**2)
    r1 = numerov(Eg_, ky1, lambda j, k, amp=amp: 0.5*(k - A_(amp)[j])**2 + Phi(amp)[j])
    print(f"amp={amp:+.1f}: max|V_before-V_after| = {np.max(np.abs(Vb-Va)):.1e};  max|t_before-t_after| = {np.nanmax(np.abs(r0['t']-r1['t'])):.1e};  max|dT| = {np.nanmax(np.abs(r0['T']-r1['T'])):.1e}")
# an unprotected channel changes
r0 = family(Eg_, 0.0, 0.0, ky1); r1 = family(Eg_, 0.0, 0.5, ky1)
print("unprotected ky=0: max|dT| =", np.nanmax(np.abs(r0['T'] - r1['T'])))

### 2. Fields before and after (Fig. B1)

In [ ]:
amp, Eres = 0.5, 0.3723
x = np.linspace(-12, 12, 4801); A0 = Ay0(x); B0 = Bz0(x); Bt = (1+amp)*B0
U_dia = -(amp + amp**2/2)*A0**2; U_par = amp*ky1*A0; U = U_dia + U_par
eE = -(amp*ky1 - (2*amp + amp**2)*A0)*B0            # eE_x = -d(e phi~)/dx
print(f"max|B| before/after = {np.abs(B0).max():.3f} / {np.abs(Bt).max():.3f};  min e*phi~ = {U.min():.3f} at x = {x[np.argmin(U)]:+.2f};  e*phi~(0) = {U[len(x)//2]:.3f};  max|eE_x| = {np.abs(eE).max():.3f}")

def stripes(ax):
    for a_, b_ in ((-4.5, -1.5), (1.5, 4.5)): ax.axvspan(a_, b_, color="#e8c46b", alpha=0.18, lw=0)
fig, ax = plt.subplots(2, 2, figsize=(12.6, 8.6))
a = ax[0, 0]; stripes(a)
a.plot(x, B0, color="0.35", lw=3, alpha=0.7, label=r"before: $B_z$"); a.plot(x, Bt, color=C_AFTER, lw=1.9, ls="--", label=r"after: $\tilde B_z=(1+\mathrm{amp})B_z$")
a.axhline(0, color="0.8", lw=0.8); a.set_xlabel("$x$"); a.set_ylabel(r"$eB_z/\hbar$"); a.legend(frameon=False, fontsize=9, loc="upper left")
a.set_title("(a) magnetic field: pure rescaling of the profile\n(shaded: the two ferromagnetic stripes)", fontsize=10.6)
a = ax[0, 1]; stripes(a)
a.plot(x, 0*x, color="0.35", lw=3, alpha=0.7, label=r"before: $e\phi=0$")
a.plot(x, U, color="#2b6cb0", lw=2, label=r"after: $e\tilde\phi=\mathrm{amp}\,k_{y1}A_y-(\mathrm{amp}+\mathrm{amp}^2/2)A_y^2$")
a.plot(x, U_dia, color="#2b6cb0", lw=1, ls=":", label=r"  diamagnetic part"); a.plot(x, U_par, color="#2b6cb0", lw=1, ls="-.", label=r"  linear part")
a.set_xlabel("$x$"); a.set_ylabel(r"$e\phi$"); a.legend(frameon=False, fontsize=8.6, loc="lower right"); a.set_title("(b) compensating gate potential: two wells over the stripes", fontsize=10.6)
a = ax[1, 0]; stripes(a)
a.plot(x, 0*x, color="0.35", lw=3, alpha=0.7, label=r"before: $E_x=0$"); a.plot(x, eE, color="#2b6cb0", lw=2, label=r"after: $eE_x=-d(e\tilde\phi)/dx$")
a.axhline(0, color="0.8", lw=0.8); a.set_xlabel("$x$"); a.set_ylabel(r"$eE_x$"); a.legend(frameon=False, fontsize=9, loc="upper left"); a.set_title("(c) electric field: odd in $x$, localized near the stripes", fontsize=10.6)
a = ax[1, 1]; stripes(a)
for ky, c in {0.0: "#2b6cb0", 0.15: "k", 0.30: C_AFTER}.items():
    a.plot(x, 0.5*(ky-A0)**2, color=c, lw=3 if ky == 0.15 else 2.2, alpha=0.45)
    a.plot(x, 0.5*(ky-(1+amp)*A0)**2 + U, color=c, lw=1.5, ls="--", label=r"$k_y=%.2f$" % ky)
a.axhline(Eres, color="0.4", lw=1, ls=":"); a.text(7.3, Eres+0.012, r"$E_{\rm res}$", fontsize=10, color="0.3")
a.set_xlabel("$x$"); a.set_ylabel(r"$V_{k_y}(x)$"); a.legend(frameon=False, fontsize=9, loc="upper right")
a.set_title("(d) 1D potentials (thick: before, dashed: after):\nidentical for $k_{y1}=0.15$, shifted by amp$(k_{y1}-k_y)A_y$ otherwise", fontsize=10.2)
fig.tight_layout(); save(fig, "fig_fields_res"); plt.show()

### 3. Resonance flow (Fig. B2)

Positions and widths of the first two transmission resonances as functions of $a$ for five values of $k_y$; $T(E)$ curves at $k_{y1}$, $0$ and $0.30$.

In [ ]:
kys5 = np.array([0.0, 0.075, 0.15, 0.225, 0.30]); amps = np.linspace(-1.0, 1.0, 21)

def peak_and_width(E, T):
    i = int(np.nanargmax(T))
    if i in (0, len(T)-1): return np.nan, np.nan, np.nan
    y0, y1, y2 = T[i-1], T[i], T[i+1]
    dd = 0.5*(y0-y2)/(y0-2*y1+y2); Epk = E[i] + dd*(E[1]-E[0]); Tpk = y1 - 0.25*(y0-y2)*dd; half = Tpk/2
    def cross(sign):
        j = i
        while 0 < j < len(T)-1 and T[j] > half: j += sign
        if j in (0, len(T)-1): return np.nan
        jj = j - sign; return E[jj] + (half - T[jj])*(E[j]-E[jj])/(T[j]-T[jj])
    return Epk, Tpk, cross(+1) - cross(-1)

def resonances0(ky, n=2):
    # rough guesses (read off T(E) at ky = 0, ky1, 0.30), interpolated in ky, then a local search for the T-maximum
    g = [np.interp(ky, [0.0, 0.15, 0.30], v) for v in ([0.2655, 0.3723, 0.5004], [0.5809, 0.6998, 0.8418])]
    out = []
    for gg in g[:n]:
        E = np.linspace(gg-0.10, gg+0.10, 4001); T = family(E, ky, 0.0, ky1)["T"]; out.append(E[np.nanargmax(T)])
    return out

flow = {}; t0 = time.time()
for ky in kys5:
    g0 = resonances0(ky)
    for nres in (0, 1):
        E = np.linspace(g0[nres]-0.14, g0[nres]+0.14, 1400)
        EE, AA = np.meshgrid(E, amps, indexing="ij")
        T = family(EE, ky, AA, ky1)["T"]
        res = np.array([peak_and_width(E, T[:, k]) for k in range(len(amps))])
        flow[(nres, ky)] = res                      # columns: E_res, T_peak, FWHM
    print(f"ky={ky:.3f}: E_res(amp=0) = {flow[(0,ky)][10,0]:.5f}, {flow[(1,ky)][10,0]:.5f};  peak T range (1st) = [{np.nanmin(flow[(0,ky)][:,1]):.5f}, {np.nanmax(flow[(0,ky)][:,1]):.5f}];  FWHM(0) = {flow[(0,ky)][10,2]:.5f}   [{time.time()-t0:.0f}s]")

In [ ]:
cols = {0.0: "#2b6cb0", 0.075: "#63b3ed", 0.15: "k", 0.225: "#ed8936", 0.30: C_AFTER}
amp_style = {-1.0: dict(color="0.6", lw=3.2, alpha=0.7, ls="-"), 0.0: dict(color="#2b6cb0", lw=1.8, ls="--"), 1.0: dict(color=C_AFTER, lw=1.8, ls=":")}
Eg = np.linspace(0.05, 1.0, 3200)
fig, ax = plt.subplots(2, 3, figsize=(15.4, 8.6))
for a, ky, ttl in ((ax[0,0], 0.15, r"(a) protected $k_{y1}=0.15$: all curves coincide"), (ax[0,1], 0.0, r"(b) $k_y=0<k_{y1}$: resonances move"), (ax[0,2], 0.30, r"(c) $k_y=0.30>k_{y1}$: other way")):
    for amp_, st in amp_style.items():
        a.plot(Eg, family(Eg, ky, amp_, ky1)["T"], label={-1.0: "amp = $-1$", 0.0: "amp = 0", 1.0: "amp = $+1$"}[amp_], **st)
    a.set_xlabel("$E$"); a.set_ylabel("$T$"); a.set_title(ttl, fontsize=10.8); a.set_xlim(0.05, 1); a.set_ylim(-0.02, 1.05)
ax[0,0].legend(frameon=False, fontsize=9, loc="lower right")
a = ax[1,0]
for ky in kys5:
    a.plot(amps, flow[(0,ky)][:,0], color=cols[ky], lw=2, label=r"$k_y=%.3f$" % ky); a.plot(amps, flow[(1,ky)][:,0], color=cols[ky], lw=2, ls="--")
a.set_xlabel("amp"); a.set_ylabel(r"$E_{\rm res}$"); a.set_title("(d) flow of the 1st (solid) and 2nd (dashed) resonance", fontsize=10.8)
a = ax[1,1]
for ky in kys5:
    for n_, ls in ((0, "-"), (1, "--")): a.plot(amps, flow[(n_,ky)][:,0] - ky**2/2, color=cols[ky], lw=2, ls=ls)
a.axvline(-1, color="0.5", lw=1, ls=":"); a.set_xlabel("amp"); a.set_ylabel(r"$E_{\rm res}-k_y^2/2$"); a.set_title("(e) lines meet at amp$=-1$: $B$ off, electrostatic twin", fontsize=10.8)
a = ax[1,2]
for ky in kys5: a.plot(amps, flow[(0,ky)][:,2], color=cols[ky], lw=2, label=r"$k_y=%.3f$" % ky)
a.axvline(-1, color="0.5", lw=1, ls=":"); a.set_xlabel("amp"); a.set_ylabel("FWHM of the 1st resonance"); a.set_title(r"(f) width flows too; frozen at $k_{y1}$", fontsize=10.8)
a.legend(frameon=False, fontsize=8.8, loc="lower left", title="transverse momentum", title_fontsize=8.8)
fig.tight_layout(); save(fig, "fig_res_1d", 190); plt.show()

In [ ]:
# ---- numbers quoted in the text ----
Eres1, Gam = flow[(0, 0.15)][10, 0], flow[(0, 0.15)][10, 2]
print(f"1st resonance at k_y1: E_res = {Eres1:.8f}, FWHM = {Gam:.5f};  2nd resonance E = {flow[(1,0.15)][10,0]:.4f}")
# <A_y>_res from the slope of the flow at small amp:  dE_res/damp = (ky1-ky) <A_y>
for nres in (0, 1):
    sl = []
    for ky in (0.0, 0.30):
        r_ = flow[(nres, ky)]; i0 = 10                            # amp=0 index; step 0.1
        sl.append((r_[i0+1, 0] - r_[i0-1, 0])/(amps[i0+1]-amps[i0-1])/(ky1 - ky))
    print(f"res #{nres+1}: <A_y>_eff from slope at amp=0: ky=0: {sl[0]:+.3f},  ky=0.30: {sl[1]:+.3f}")
# |psi|^2-weighted average over the barrier region at the resonance (amp=0, ky=ky1)
rr = family(Eres1, ky1, 0.0, ky1, want_psi=True); w_ = np.abs(rr["psi"][:, ...])**2
for xlim in (3.0, 4.5, 6.0):
    msk = np.abs(xs) < xlim; print(f"  |psi|^2-weighted <A_y> over |x|<{xlim}: {np.sum(A0g[msk]*w_[msk])/np.sum(w_[msk]):+.3f}")
print("peak height T at resonance: min/max over all (ky, amp):",
      min(np.nanmin(v[:,1]) for v in flow.values()), max(np.nanmax(v[:,1]) for v in flow.values()))
# amp=-1 twin: T depends on E - ky^2/2 only
Et = 0.37; kk = ky1; dk_ = 1e-4
lnT = lambda E_, k_: np.log(family(E_, k_, -1.0, ky1)["T"])
dky = (lnT(Et, kk+dk_) - lnT(Et, kk-dk_))/(2*dk_)
dE_ = (np.log(family(Et+dk_, kk, 0.0, ky1)["T"]) - np.log(family(Et-dk_, kk, 0.0, ky1)["T"]))/(2*dk_)
print(f"amp=-1: d ln T~/dky = {float(dky):+.4f};   -ky1 * d ln T/dE (original, protected channel) = {float(-ky1*dE_):+.4f}")

### 4. Maps $T(E,k_y)$ (Fig. B3)

In [ ]:
ampM = 0.5
Eg_m = np.linspace(0.02, 1.0, 400 if FAST else 1000); kg_m = np.linspace(-1.0, 1.0, 161 if FAST else 401)
EEm, KYm = np.meshgrid(Eg_m, kg_m, indexing="ij"); t0 = time.time()
T0m = family(EEm, KYm, 0.0, ky1)["T"]; T1m = family(EEm, KYm, ampM, ky1)["T"]; print("maps done in %.0f s" % (time.time()-t0))
dT = T1m - T0m; i = np.argmin(abs(kg_m - ky1))
print("max|dT| on the protected column:", np.nanmax(np.abs(dT[:, i])), "   global max|dT|:", np.nanmax(np.abs(dT)))

In [ ]:
KYp, EEp = np.meshgrid(kg_m, Eg_m)
def style(ax, title):
    ax.set_facecolor("0.88"); ax.plot(kg_m, kg_m**2/2, color="0.35", lw=1, ls="--")
    ax.set_xlim(-1, 1); ax.set_ylim(0.02, 1.0); ax.axvline(ky1, color="k", lw=1, ls=":", alpha=0.8)
    ax.set_xlabel(r"canonical $k_y$"); ax.set_ylabel("$E$"); ax.set_title(title, fontsize=10.6)
fig, ax = plt.subplots(2, 3, figsize=(17.6, 10.6))
a = ax[0,0]; m_ = a.pcolormesh(KYp, EEp, np.ma.masked_invalid(T0m), cmap="viridis", vmin=0, vmax=1, shading="auto", rasterized=True); style(a, r"(a) $T(E,k_y)$, original double barrier"); fig.colorbar(m_, ax=a, pad=0.02).set_label("$T$")
a = ax[0,1]; m_ = a.pcolormesh(KYp, EEp, np.ma.masked_invalid(dT), cmap="RdBu_r", norm=TwoSlopeNorm(0, -1, 1), shading="auto", rasterized=True); style(a, r"(b) $\Delta T$ (amp$=%+.1f$), linear" % ampM); fig.colorbar(m_, ax=a, pad=0.02).set_label(r"$\Delta T$")
a = ax[0,2]; m_ = a.pcolormesh(KYp, EEp, np.ma.masked_invalid(np.abs(dT) + 1e-16), cmap="magma_r", norm=LogNorm(1e-13, 1), shading="auto", rasterized=True); style(a, r"(c) $|\Delta T|$, log scale (pale = saturation, not protection)"); fig.colorbar(m_, ax=a, pad=0.02).set_label(r"$|\Delta T|$")
R0, R1 = 1-T0m, 1-T1m
with np.errstate(all="ignore"): D = np.where(T0m < 0.5, np.abs(np.log(T1m/T0m)), np.abs(np.log(R1/R0)))
bad = ~np.isfinite(D) | (R0 < 1e-10) | (R1 < 1e-10) | (T0m < 1e-12) | (T1m < 1e-12); D = np.ma.masked_where(bad, D)
a = ax[1,0]; m_ = a.pcolormesh(KYp, EEp, D, cmap="viridis", norm=LogNorm(1e-3, 10), shading="auto", rasterized=True); style(a, r"(d) relative change $|\Delta\ln T|$ ($T<1/2$) or $|\Delta\ln R|$: dark stripe = protected"); fig.colorbar(m_, ax=a, pad=0.02)
a = ax[1,1]
for T, col, lab, s, al in ((T0m, "0.45", "before", 12, 0.55), (T1m, C_AFTER, "after", 4, 0.9)):
    X, Y = [], []
    for j, ky in enumerate(kg_m):
        cT = T[:, j]
        if np.isnan(cT).all(): continue
        pk, _ = find_peaks(np.nan_to_num(cT), height=0.9, prominence=0.04); X += [ky]*len(pk); Y += list(Eg_m[pk])
    a.scatter(X, Y, s=s, color=col, label=lab, alpha=al, linewidths=0)
style(a, r"(e) positions of the $T$-maxima before/after"); a.legend(frameon=False, loc="lower right", markerscale=1.6)
a = ax[1,2]; cmv = plt.get_cmap("viridis"); Es_sl = [0.25, 0.372, 0.50, 0.70, 0.90]
for n_, Es in enumerate(Es_sl):
    ii = np.argmin(abs(Eg_m - Es)); a.plot(kg_m, dT[ii], lw=1.7, color=cmv(n_/(len(Es_sl)-1)), label=r"$E=%.3f$" % Eg_m[ii])
a.axvline(ky1, color="k", lw=1, ls=":"); a.axhline(0, color="0.7", lw=0.8); a.set_xlim(-0.2, 0.6); a.set_ylim(-1.05, 1.05)
a.set_xlabel(r"canonical $k_y$"); a.set_ylabel(r"$\Delta T$"); a.set_title(r"(f) slices at fixed $E$: all vanish at $k_{y1}$", fontsize=10.6); a.legend(frameon=False, fontsize=9, loc="upper right")
fig.tight_layout(); save(fig, "fig_maps_res", 165); plt.show()

### 5. Resonant beam (Fig. B4)

$\Psi(x,y)=\int g(k_y)\psi_{k_y}(x)e^{ik_yy}dk_y$ with $g=\sqrt{W/k_x}$, $W=e^{-(k_y-k_{y1})^2/2\sigma_k^2}$ the incident flux, at $E=E_{\rm res}(k_{y1})$ (the same before and after).

In [ ]:
def beam(E, sig, ampb, ky1=ky1, nk=161, span=4.2, want_fields=True, A_stats_only=False):
    kys = ky1 + sig*np.linspace(-span, span, nk); dk = kys[1]-kys[0]; kx = np.sqrt(2*E - kys**2)
    W = np.exp(-(kys-ky1)**2/(2*sig**2)); g = np.sqrt(W/kx)
    r0 = family(E, kys, 0.0, ky1, want_psi=want_fields); r1 = family(E, kys, ampb, ky1, want_psi=want_fields)
    return dict(kys=kys, dk=dk, kx=kx, W=W, g=g, r0=r0, r1=r1)

def frac(bm, which):
    T = bm[which]["T"]; wT = bm["W"]*T
    return wT.sum()/bm["W"].sum(), (bm["kys"]*wT).sum()/wT.sum()

E_b, sig_b, amp_b = Eres1, 0.02, 0.5
bm = beam(E_b, sig_b, amp_b)
print(f"E = {E_b:.6f}, sigma_k = {sig_b}: transmitted fraction before {frac(bm,'r0')[0]:.4f}, after {frac(bm,'r1')[0]:.4f}")

# total transmitted fraction vs sigma_k (amp = 0.5)
print("sigma_k   dT_tot/T_tot (%)")
for s_ in (0.002, 0.004, 0.008, 0.012, 0.02, 0.04):
    b_ = beam(E_b, s_, amp_b, want_fields=False); f0_, f1_ = frac(b_, 'r0')[0], frac(b_, 'r1')[0]
    print(f"{s_:7.3f}   {100*(f1_/f0_-1):+7.1f}")

In [ ]:
sub = 5
xs_a = bm["r0"]["xs"][::-1][::sub]; ys = np.linspace(-120, 340, 920)
phase = np.exp(1j*np.outer(bm["kys"], ys))
synth = lambda psi: ((psi[::-1][::sub])*bm["g"][None, :]) @ phase * bm["dk"]
I0 = np.abs(synth(bm["r0"]["psi"]))**2; I1 = np.abs(synth(bm["r1"]["psi"]))**2
If = np.abs((np.exp(1j*np.outer(xs_a, bm["kx"]))*bm["g"][None, :]) @ phase * bm["dk"])**2; Imax = If.max()
xshow = 26.0; ix = np.argmin(abs(xs_a - xshow))
mom = lambda p: ((ys*p).sum()/p.sum(), np.sqrt(((ys-(ys*p).sum()/p.sum())**2*p).sum()/p.sum()))
cf, sf = mom(If[ix]); c0, s0 = mom(I0[ix]); c1, s1 = mom(I1[ix])
print(f"transmitted beam at x={xshow:.0f}: centroid free {cf:+.1f} (rms {sf:.0f}) | before {c0:+.1f} (rms {s0:.0f}) | after {c1:+.1f} (rms {s1:.0f})")

fig = plt.figure(figsize=(13.4, 10.2)); gs = fig.add_gridspec(2, 2, hspace=0.33, wspace=0.24)
ax = fig.add_subplot(gs[0,0]); kys_, W_ = bm["kys"], bm["W"]
ax.fill_between(kys_, W_/W_.max(), color="0.82", label=r"incident flux $W(k_y)$")
ax.plot(kys_, bm["r0"]["T"], color="0.45", lw=1.4, ls=":", label=r"$T(k_y)$, before"); ax.plot(kys_, bm["r1"]["T"], color=C_AFTER, lw=1.2, ls=":", label=r"$\tilde T(k_y)$, after")
ax.plot(kys_, W_*bm["r0"]["T"]/W_.max(), color="0.2", lw=2.8, alpha=0.65, label=r"transmitted, before"); ax.plot(kys_, W_*bm["r1"]["T"]/W_.max(), color=C_AFTER, lw=1.9, ls="--", label="transmitted, after")
ax.axvline(ky1, color="k", lw=1.2, ls=":"); ax.set_xlabel(r"canonical $k_y$"); ax.set_ylabel("flux per $dk_y$ / $T$"); ax.set_ylim(-0.02, 1.08)
ax.set_title(r"(a) $k_y$-content at the resonance energy ($E=%.4f$, $\sigma_k=%.2f$)" % (E_b, sig_b), fontsize=10.8); ax.legend(frameon=False, fontsize=8.6, loc="upper right")
ax = fig.add_subplot(gs[0,1])
ax.plot(ys, If[ix]/Imax, color="0.6", lw=1.6, label=f"free beam (centre {cf:+.0f})"); ax.plot(ys, I0[ix]/Imax, color="0.2", lw=3, alpha=0.6, label=f"before (centre {c0:+.0f}, rms {s0:.0f})"); ax.plot(ys, I1[ix]/Imax, color=C_AFTER, lw=1.9, ls="--", label=f"after (centre {c1:+.0f}, rms {s1:.0f})")
ax.set_xlabel("$y$"); ax.set_ylabel(r"$|\Psi(x{=}%.0f,y)|^2$ / free-beam peak" % xshow); ax.set_title(r"(b) transmitted beam at $x=%.0f$" % xshow, fontsize=10.8); ax.legend(frameon=False, fontsize=8.8, loc="upper right"); ax.set_xlim(-100, 340)
for k, (I, ttl) in enumerate(((I0, "(c) before"), (I1, r"(d) after ($\mathrm{amp}=%+.1f$)" % amp_b))):
    ax = fig.add_subplot(gs[1, k]); im = ax.imshow(I.T/Imax, origin="lower", extent=[xs_a[0], xs_a[-1], ys[0], ys[-1]], aspect="auto", cmap="magma", norm=PowerNorm(0.4, vmin=0, vmax=1.5))
    ax.axvspan(-6, 6, color="#4aa3ff", alpha=0.10); xg = np.array([xs_a[0], xs_a[-1]]); ax.plot(xg, xg*ky1/np.sqrt(2*E_b-ky1**2), color="w", lw=0.8, ls=":")
    ax.set_xlabel("$x$"); ax.set_ylabel("$y$"); ax.set_title(ttl + r": $|\Psi(x,y)|^2$", fontsize=10.6)
fig.colorbar(im, ax=ax, pad=0.02, extend="max").set_label(r"$|\Psi|^2/I_{\rm free,max}$ (gamma 0.4)")
save(fig, "fig_beam_res", 170); plt.show()

In [ ]:
# ---- transverse drift of the resonance and the lateral (Goos-Haenchen-like) shift ----
dk_ = 0.005
def Eres_at(ky, amp_):
    E = np.linspace(Eres1-0.12, Eres1+0.12, 4801); T = family(E, ky, amp_, ky1)["T"]; return peak_and_width(E, T)[0]
print("amp    dE_res/dky   d arg t/dky (E=E_res, ky=ky1)   -2 E'/Gamma")
for amp_ in (0.0, 0.5, -0.5):
    Ep = (Eres_at(ky1+dk_, amp_) - Eres_at(ky1-dk_, amp_))/(2*dk_)
    tp = family(Eres1, ky1+dk_, amp_, ky1)["t"]; tm = family(Eres1, ky1-dk_, amp_, ky1)["t"]
    darg = np.angle(tp/tm)/(2*dk_)
    print(f"{amp_:+.1f}   {Ep:9.4f}   {float(darg):12.1f}              {-2*Ep/Gam:9.1f}")

### 6. Comparison: a single non-resonant stripe (numbers quoted in the text)

In [ ]:
# single finite stripe (zero net flux): A_s(x) = A1(x+Lx/2) - A1(x-Lx/2)
As = lambda x: A1(x + Lx/2) - A1(x - Lx/2)
A1g = As(xs)
def family_single(E, ky, amp, ky1, want_psi=False):
    E, ky, amp = np.broadcast_arrays(np.asarray(E, float), np.asarray(ky, float), np.asarray(amp, float))
    return numerov(E, ky, lambda j, kyr: 0.5*(kyr - A1g[j])**2 + amp.ravel()*(ky1 - kyr)*A1g[j], want_psi)

E_s, sig_s, nk = 0.4, 0.10, 121
kys = ky1 + sig_s*np.linspace(-4.2, 4.2, nk); dk = kys[1]-kys[0]; kx = np.sqrt(2*E_s - kys**2)
Ws = np.exp(-(kys-ky1)**2/(2*sig_s**2)); gs_ = np.sqrt(Ws/kx)
q0 = family_single(E_s, kys, 0.0, ky1, True); q1 = family_single(E_s, kys, 0.5, ky1, True)
ang = lambda k: np.degrees(np.arcsin(k/np.sqrt(2*E_s)))
for nm, q in (("before", q0), ("after", q1)):
    wT = Ws*q["T"]; km = (kys*wT).sum()/wT.sum(); print(f"{nm}: transmitted fraction {wT.sum()/Ws.sum():.4f}, mean angle {ang(km):.2f} deg")
ys2 = np.linspace(-30, 42, 360); ph2 = np.exp(1j*np.outer(kys, ys2)); xs2 = q0["xs"][::-1][::5]
def centroid_at(q, x0):
    P = (q["psi"][::-1][::5]*gs_[None, :]) @ ph2 * dk; I = np.abs(P)**2; i = np.argmin(abs(xs2-x0)); return (ys2*I[i]).sum()/I[i].sum()
print("centroid at x = 28:  before %+.2f   after %+.2f" % (centroid_at(q0, 28.0), centroid_at(q1, 28.0)))
xx_ = np.linspace(14, 28, 8)
for nm, q in (("before", q0), ("after", q1)):
    yc = np.array([centroid_at(q, x_) for x_ in xx_]); s_, b_ = np.polyfit(xx_, yc, 1); print(f"{nm}: beam axis y_c = {b_:+.2f} + {s_:.4f} x  ({np.degrees(np.arctan(s_)):.2f} deg)")

### 7. Gate realization and tolerances (Fig. B5)

The ideal compensating potential $e\tilde\phi(x)$ is fitted by a few left–right symmetric, Gaussian-screened electrodes (least squares); we then compute how far the resonance of the protected channel moves.

In [ ]:
ampG = 0.5; AtG = (1+ampG)*A0g; U_id = ampG*ky1*A0g - (ampG + ampG**2/2)*A0g**2
smooth = lambda U, s: gaussian_filter1d(U, s/h_, mode="nearest") if s > 0 else U
def solve_U(E, At, U): return numerov(E, ky1, lambda j, kyr: 0.5*(kyr - At[j])**2 + U[j])["T"]
def peak_pos(E, T):
    i = np.nanargmax(T)
    if i in (0, len(T)-1): return np.nan
    y0, y1, y2 = T[i-1], T[i], T[i+1]; return E[i] + 0.5*(y0-y2)/(y0-2*y1+y2)*(E[1]-E[0])
Efine = np.linspace(0.31, 0.43, 2401); Ewide = np.linspace(0.05, 1.0, 1500); Ewres = np.linspace(0.10, 0.70, 6001)
T0f = solve_U(Efine, A0g, 0*A0g); E0 = peak_pos(Efine, T0f); T0w = solve_U(Ewide, A0g, 0*A0g)

def evaluate(name, U, verbose=True):
    Tf = solve_U(Efine, AtG, U); pk = peak_pos(Efine, Tf)
    if np.isnan(pk): Tr = solve_U(Ewres, AtG, U); pk = Ewres[np.nanargmax(Tr)]
    Tw = solve_U(Ewide, AtG, U); far = np.abs(Ewide - E0) > 3*Gam
    dE = pk - E0
    if verbose: print(f"{name:48s} dE_res = {dE:+.5f} ({dE/Gam:+.2f} Gamma)   max|dT| away from the resonance = {np.nanmax(np.abs(Tw-T0w)[far]):.4f}")
    return dE

print(f"baseline: E_res = {E0:.5f}, Gamma = {Gam:.5f}")
evaluate("ideal compensation", U_id)
for eps in (0.01, -0.01, 0.05, -0.05): evaluate(f"amplitude error {eps:+.2f}", (1+eps)*U_id)
for s in (0.25, 0.5, 1.0): evaluate(f"ideal profile smoothed, s={s}", smooth(U_id, s))
stripe = ((xs > -4.5) & (xs < -1.5)) | ((xs > 1.5) & (xs < 4.5)); F = smooth(stripe.astype(float), 0.5); c_ = (F*U_id).sum()/(F*F).sum()
evaluate("gate only over the stripes (fitted amplitude), s=0.5", c_*F)

In [ ]:
def regions(ed): return [((np.abs(xs) >= a_) & (np.abs(xs) < b_)).astype(float) for a_, b_ in ed]
def fit_model(ed, s):
    Mx = np.array([smooth(r_, s) for r_ in regions(ed)]).T; msk = np.abs(xs) < 14
    coef, *_ = np.linalg.lstsq(Mx[msk], U_id[msk], rcond=None); return Mx @ coef, coef
models = {"2 electrodes": [(0,1.5),(1.5,4.5)], "3 electrodes": [(0,1.5),(1.5,4.5),(4.5,8.0)],
          "6 electrodes": [(0,1.5),(1.5,3.0),(3.0,4.5),(4.5,6.0),(6.0,8.0),(8.0,14.0)]}
mcol = {"2 electrodes": "#ed8936", "3 electrodes": "#2b6cb0", "6 electrodes": "#2f855a"}
Us = {}
for nm, ed in models.items():
    Us[nm], coef = fit_model(ed, 0.5); evaluate(nm + " (Gaussian screening s=0.5)", Us[nm]); print("      voltages:", np.round(coef, 3))
fig, ax = plt.subplots(1, 2, figsize=(13.2, 4.8)); a = ax[0]
for a_, b_ in ((-4.5, -1.5), (1.5, 4.5)): a.axvspan(a_, b_, color="#e8c46b", alpha=0.18, lw=0)
a.plot(xs, U_id, color="k", lw=3, alpha=0.5, label=r"ideal $e\tilde\phi(x)$")
for nm, U_ in Us.items(): a.plot(xs, U_, color=mcol[nm], lw=1.6, ls="--", label=nm)
a.set_xlim(-12, 12); a.set_xlabel("$x$"); a.set_ylabel(r"$e\phi$"); a.legend(frameon=False, fontsize=9, loc="lower right"); a.set_title("(a) ideal compensating potential and least-squares fits", fontsize=10.6)
a = ax[1]; Ef = np.linspace(0.34, 0.41, 1401)
a.plot(Ef, solve_U(Ef, A0g, 0*A0g), color="k", lw=3.2, alpha=0.5, label="original = ideal after")
for nm, U_ in Us.items(): a.plot(Ef, solve_U(Ef, AtG, U_), color=mcol[nm], lw=1.6, ls="--", label=nm)
a.plot(Ef, solve_U(Ef, AtG, 0*A0g), color=C_AFTER, lw=1.3, ls=":", label=r"no gate ($e\tilde\phi=0$)")
a.set_xlabel("$E$"); a.set_ylabel(r"$T(E,k_{y1})$"); a.set_title(r"(b) the protected resonance at $k_{y1}=0.15$", fontsize=10.6); a.legend(frameon=False, fontsize=9, loc="upper left")
fig.tight_layout(); save(fig, "fig_gates"); plt.show()

---
**Things to try.** Change `b, d, H, Lx, gap` (barrier), `ky1` (protected channel), `amp` (deformation) and re-run Part IV; set `FAST=True` for coarse maps; `SAVE=True` writes the figures.